## Imports

In [1]:
import pandas as pd
import numpy as np

## Data Profiling

In [2]:
users = pd.read_csv("../data/users.csv")
print(users.shape)

pd.DataFrame({
    "dtype": users.dtypes,
    "null_count": users.isna().sum(),
    "unique_counts": users.nunique()
})

(20000, 4)


,dtype,null_count,unique_counts
user_id,int64,0,20000
signup_date,str,0,540
city,str,0,6
acquisition_channel,str,0,4


In [3]:
orders = pd.read_csv("../data/orders.csv")
print(orders.shape)

pd.DataFrame({
    "dtype": orders.dtypes,
    "null_count": orders.isna().sum(),
    "unique_counts": orders.nunique()
})

(96020, 9)


,dtype,null_count,unique_counts
order_id,int64,0,96020
user_id,int64,0,17161
store_id,str,0,48
city,str,0,6
order_ts,str,0,85953
items,int64,0,15
gmv,float64,0,57314
delivery_min,float64,0,366
category,str,0,6


In [4]:
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery


In [5]:
store_city_mapping = orders[["city", "store_id"]].drop_duplicates()
store_city_mapping = store_city_mapping.sort_values(by = ["city", "store_id"], ascending = [True, True])
store_city_mapping

,city,store_id
48,Bengaluru,BEN-01
34,Bengaluru,BEN-02
15,Bengaluru,BEN-03
97,Bengaluru,BEN-04
38,Bengaluru,BEN-05
19,Bengaluru,BEN-06
4,Bengaluru,BEN-07
49,Bengaluru,BEN-08
156,Delhi,DEL-01
73,Delhi,DEL-02


In [6]:
experiment = pd.read_csv("../data/experiment.csv")
print(experiment.shape)

pd.DataFrame({
    "dtype": experiment.dtypes,
    "null_count": experiment.isna().sum(),
    "unique_counts": experiment.nunique()
})

(12000, 6)


,dtype,null_count,unique_counts
user_id,int64,0,12000
variant,str,0,2
exposure_date,str,0,14
pre_gmv_28d,float64,0,8662
orders_14d,int64,0,19
gmv_14d,float64,0,7130


In [7]:
users['signup_date'] = pd.to_datetime(users['signup_date'])
orders['order_ts'] = pd.to_datetime(orders['order_ts'])
experiment['exposure_date'] = pd.to_datetime(experiment['exposure_date'])

In [8]:
# Runtime of experiment
print(f"Experiment ran from {experiment['exposure_date'].min().date()} till {experiment['exposure_date'].max().date()}")

Experiment ran from 2026-09-01 till 2026-09-14


In [9]:
# Orders data
print(f"Orders data is from {orders['order_ts'].min().date()} till {orders['order_ts'].max().date()}")

Orders data is from 2025-01-02 till 2026-09-29


In [10]:
# Users signup dates
print(f"Users belongs to {users['signup_date'].min().date()} till {users['signup_date'].max().date()} date range")

Users belongs to 2025-01-01 till 2026-06-24 date range


In [11]:
users_rows = len(users)
unique_users = users['user_id'].nunique()
print(f"Users base has {unique_users} users")

if users_rows == unique_users:
    print("This ensures uniqueness of user_id in users base")
else:
    print("Users base user_id isn't unique")

Users base has 20000 users
This ensures uniqueness of user_id in users base


In [12]:
experiment_rows = len(experiment)
unique_experiment_users = experiment['user_id'].nunique()
print(f"Experiments base has {unique_experiment_users} users")

if experiment_rows == unique_experiment_users:
    print("This ensures uniqueness of user_id in experiment base")
else:
    print("Experiment base user_id isn't unique")

Experiments base has 12000 users
This ensures uniqueness of user_id in experiment base


In [13]:
print(f"Number of users in orders: {orders['user_id'].nunique()}")

Number of users in orders: 17161


In [14]:
orders = orders.merge(users, on = "user_id", how = "left", suffixes = ("_orders", "_users"))

acquisition_order_summary = (
    orders
    .groupby(["acquisition_channel"], as_index = False)
    .agg(
        users = ("user_id", "nunique"),
        orders = ("order_id", "count"),
        gmv = ("gmv", "sum")
    )
)

acquisition_order_summary["order_frequency"] = acquisition_order_summary["orders"]/acquisition_order_summary["users"]
acquisition_order_summary["average_order_value"] = acquisition_order_summary["gmv"]/acquisition_order_summary["users"]

acquisition_order_summary

,acquisition_channel,users,orders,gmv,order_frequency,average_order_value
0,organic,7751,43333,18565754.73,5.590633,2395.272188
1,paid_social,4282,23869,10259255.06,5.574264,2395.902630
2,referral,2543,14251,6082651.72,5.604011,2391.919670
3,search,2585,14567,6219598.48,5.635203,2406.034228


In [15]:
len(orders[orders["city_orders"] != orders["city_users"]])

0

In [16]:
city_order_summary = (
    orders
    .groupby(["city_orders"], as_index = False)
    .agg(
        users = ("user_id", "nunique"),
        orders = ("order_id", "count"),
        gmv = ("gmv", "sum")
    )
)

city_order_summary["order_frequency"] = city_order_summary["orders"]/city_order_summary["users"]
city_order_summary["average_order_value"] = city_order_summary["gmv"]/city_order_summary["users"]

city_order_summary

,city_orders,users,orders,gmv,order_frequency,average_order_value
0,Bengaluru,4223,24173,10392722.49,5.724130,2460.980935
1,Delhi,4177,22899,9781073.77,5.482164,2341.650412
2,Gurugram,1759,9746,4168862.16,5.540648,2370.018283
3,Hyderabad,1762,9878,4260769.00,5.606129,2418.143587
4,Mumbai,3483,19404,8297482.06,5.571059,2382.280235
5,Pune,1757,9920,4226350.51,5.645987,2405.435692


In [17]:
print(orders['gmv'].quantile([0.5, 0.9, 0.99, 1]))

0.50     343.7850
0.90     817.4610
0.99    1620.5149
1.00    5032.8300
Name: gmv, dtype: float64


In [18]:
print(orders['delivery_min'].quantile([0.5, 0.9, 0.99, 1]))

0.50    15.3
0.90    22.5
0.99    30.3
1.00    45.3
Name: delivery_min, dtype: float64


In [19]:
store_summary = orders.groupby('store_id').agg(
    order_count=('order_id', 'count'),          # Number of orders (assumes 'order_id' is your primary key)
    total_gmv=('gmv', 'sum'),                    # Total Gross Merchandise Value
    unique_users=('user_id', 'nunique'),         # Number of unique users
    p50_delivery=('delivery_min', lambda x: x.quantile(0.50)),
    p90_delivery=('delivery_min', lambda x: x.quantile(0.90)),
    p99_delivery=('delivery_min', lambda x: x.quantile(0.99)),
    max_delivery=('delivery_min', 'max')         # 1.0 quantile is just the maximum value
)

store_summary

,order_count,total_gmv,unique_users,p50_delivery,p90_delivery,p99_delivery,max_delivery
store_id,,,,,,,
BEN-01,3017,1257483.94,1928,15.1,22.30,30.200,38.2
BEN-02,3114,1322807.08,1973,15.3,22.57,30.300,42.9
BEN-03,3033,1310569.50,1903,15.5,22.70,31.068,45.3
BEN-04,3013,1310135.24,1908,15.4,22.50,29.988,42.8
BEN-05,2981,1301647.30,1935,15.5,22.50,30.420,44.3
BEN-06,3121,1369130.47,1978,15.4,22.60,30.980,39.7
BEN-07,2939,1261571.60,1874,15.4,22.72,30.748,42.8
BEN-08,2955,1259377.36,1926,15.4,22.76,29.500,36.5
DEL-01,2870,1229749.16,1852,15.6,22.51,30.731,40.8


In [20]:
orders_rows = len(orders)
unique_orders = orders['order_id'].nunique()
print(f"Orders base has {unique_orders} orders")

if orders_rows == unique_orders:
    print("This ensures uniqueness of order_id in orders base")
else:
    print("Orders base order_id isn't unique")

Orders base has 96020 orders
This ensures uniqueness of order_id in orders base


Number of orders in orders table is 96020 and we are able to deliver 99% of them in 30 minutes

### Things I'd ask the data owner about
1. pre_gmv_28d, orders_14d, gmv_14d - are these related to exposure_date? if yes then what time?
2. order_ts column, is this UTC or IST?
3. How come we are getting more orders from Organic channel instead of pain one? Is this correct? If yes, then we might need to work on marketing.
4. Each city has 8 stores, even though we are getting significantly more orders from Bengaluru and Delhi. Is there any plan to open more stores in these cities?
5. All of our stores are delivering 99% of orders within 30 minutes, that's good performance. Are we sure, delivery done is being marked only after delivered to customer. Riders aren't marked delivered by themselves to reach the targets?